# CCAR-F — Claude Certified Architect – Foundations · Homework (full-length mock exam)

**Goal:** sit a full-length, original mock in the CCAR-F format (4 scenarios, 60 items, 120 minutes), then read a score report like the real one: percent correct per domain, a scaled-score estimate, and PASS/RETRY at 720.

- Overview, blueprint and study plan: [README.md](README.md) · Drills first: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: **120 minutes** for the exam, plus 45–60 minutes to review the explanations
- Estimated API cost: **$0**. Everything runs offline; no API key is needed.
- Prerequisites: the 8-week [study plan](README.md#8-week-study-plan-using-this-repo), or at least the chapters for all five domains

**Exam conditions**
1. One sitting, closed book: no docs, no notes, no Claude, no search.
2. Run the **timer** cell, then read Scenario A. Budget about 30 minutes per scenario (2 minutes per item).
3. Write every answer into the single `answers` dict below the four scenarios. Use one letter for normal items, and the stated number of letters for *Select TWO* items (`"BD"`, `"B,D"` or `["B", "D"]`).
4. Run the grader. Open the explanations at the bottom **only after** grading.

**How this mock is built.** The real exam draws 4 of the 6 official scenarios. This mock uses Scenario A (customer support, SC1), B (code generation with Claude Code, SC2), C (multi-agent research, SC3) and D (Claude Code in CI, SC5), because that combination reproduces the official domain weights: 16 / 11 / 12 / 12 / 9 items for Domains 1–5 (27 / 18 / 20 / 20 / 15%). SC4 and SC6 are covered in the [drills](01_practice.ipynb). Eight items are multiple-response. All items are **original**: none comes from the guide's sample questions or the live exam. Multiple-response items are scored all-or-nothing, and keys are stored as SHA-256 hashes.

**Pass mark.** The real exam reports a scaled score (100–1,000, pass = 720). Anthropic does not publish its raw-to-scaled conversion, so the grader uses a documented **linear approximation** that maps 72% raw to 720. Treat the estimate as a rough signal, not a prediction.

## Setup and timer

In [ ]:
import hashlib
import time
from datetime import datetime, timedelta

try:
    from IPython.display import Markdown, display
except ImportError:  # plain Python: fall back to text
    Markdown = display = None

print("Offline mock exam: no API key or network needed.")

Run this cell when you are ready to start. It records the start time; the grader reports how long you took.

In [ ]:
EXAM_START = time.time()
EXAM_MINUTES = 120
end = datetime.now() + timedelta(minutes=EXAM_MINUTES)
print(f"Started {datetime.now():%H:%M}. Time is up at {end:%H:%M}. Checkpoints: A by +30, B by +60, C by +90, D by +120 minutes.")

---

## Scenario A: Customer support resolution agent (Q1–Q15)

*Based on official scenario SC1.*

You run the support agent for **Larkspur Mobile**, a fictional mobile carrier. The agent is built with the **Claude Agent SDK** and reaches the billing and CRM back ends through a custom **MCP server** with these tools: `verify_identity`, `get_account`, `get_invoice`, `get_billing_history`, `apply_credit`, `change_plan` and `escalate_to_human`.

Company policy: identity must be verified before any account change, and credits above **$200** need a supervisor. Human agents work in a separate ticketing system and **cannot see the chat transcript**. The business target is **75% first-contact resolution** without wrongful credits.

**Q1.** Despite a system-prompt rule, audit logs show that last month the agent applied eleven credits between $240 and $410 without supervisor approval. You need a guarantee that this cannot happen again, while these customers still get a path to resolution. What should you implement?

- A) A `PreToolUse` hook on `apply_credit` that denies amounts above $200 and tells the agent to call `escalate_to_human`.
- B) Restate the $200 limit in bold at the start and end of the system prompt, so the rule cannot get lost in the middle of a long context.
- C) A `PostToolUse` hook on `apply_credit` that sends every credit above $200 to a supervisor audit queue for review.
- D) Add few-shot examples in which the agent escalates every credit request above $200, so the model learns the boundary from demonstrations.

**Q2.** *(Select TWO)* About 4% of `change_plan` and `apply_credit` calls happen before `verify_identity` has succeeded, mostly on chats handed over from the IVR phone menu with an account number already attached. Which TWO changes make verification a hard prerequisite?

- A) Have the agent state its confidence that the caller is verified before each account change, and block changes below 0.8.
- B) Add a bold 'ALWAYS run verify_identity first, even when an account number arrives from the IVR' rule to the system prompt.
- C) Record in application state when `verify_identity` succeeds, and have a `PreToolUse` hook deny the two tools until it is set.
- D) Make the two tools require a token that only a successful `verify_identity` call issues, and reject calls without one.
- E) Add few-shot examples of IVR handovers in which the agent still runs `verify_identity` before touching the account.

**Q3.** For the SMS channel the team wrote a lightweight agentic loop directly on the Messages API: call the API; if the response contains a text block, send that text to the customer and stop; otherwise run the requested tools and call again. Testers report that the agent often stops right after saying 'Let me pull up your invoice' and never shows the invoice. What is the correct fix?

- A) Always run exactly three iterations, which covers the usual verify, look-up and answer path for SMS requests.
- B) Add 'Never write any text before calling a tool' to the system prompt, so a text block reliably means the answer is final.
- C) Stop only when the text contains a closing phrase such as 'anything else I can help with?', which marks a finished answer.
- D) Loop while `stop_reason` is `"tool_use"` (run tools, append `tool_result` blocks, call again); stop on `"end_turn"`.

**Q4.** Escalated customers wait about 7 minutes while the human agent re-asks for account details and the history of the issue, because the ticketing system receives only the text 'Customer needs help with billing.' What is the best change?

- A) Include a link to the full chat transcript in the escalation, so the human agent can read the whole conversation history before replying.
- B) Make `escalate_to_human` carry a structured handoff (account, verification, root cause, actions taken, amount, recommendation).
- C) Have the agent tell the customer to summarize the issue again for the human agent, so the details come first-hand.
- D) Escalate earlier in the conversation, so there is less history to pass on and the human agent can take over sooner.

**Q5.** A customer writes: 'I was charged twice in August, roaming hasn't worked since I landed in Lisbon, and I'd like to move to a cheaper plan.' The agent fixes the double charge and ends the conversation. Which design gives the best outcome?

- A) Split the message into three issues, investigate each with the shared account facts (in parallel where possible), and reply once.
- B) Handle only the most urgent issue and list the others at the end of the reply so the customer can raise them later.
- C) Ask the customer to open a separate conversation for each issue, so that each one gets the agent's full attention.
- D) Escalate every message that contains more than one issue to a human agent, because multi-issue requests are too complex to automate reliably.

**Q6.** `get_invoice` returns due dates as Unix timestamps, `get_billing_history` returns dates like `04/09/2026`, and both return payment status as numeric codes (1, 2, 7). The agent sometimes miscalculates late fees and misreads statuses. What is the most reliable fix?

- A) A `PostToolUse` hook that converts dates to ISO 8601 and status codes to named statuses before the agent reads results.
- B) Instruct the agent to convert all dates to one format step by step in its reasoning before it calculates late fees.
- C) A table in the system prompt that explains every date format and status code the two tools can return.
- D) Route every tool result through a dedicated normalization subagent before the main agent sees it, to keep the main prompt short.

**Q7.** `get_account` and `get_invoice` now have detailed, clearly differentiated descriptions, yet the agent still calls `get_account` for most invoice questions. The system prompt contains: 'Whenever the customer mentions their bill, check their account first.' What should you do first?

- A) Add a lightweight routing classifier that reads each message and picks the right tool before the agent's turn begins.
- B) Rewrite that system-prompt instruction, which links the word 'bill' to `get_account` and overrides the descriptions.
- C) Merge the two tools into a single `get_billing_info` tool, so the agent no longer has to choose between two similar tools.
- D) Force `get_invoice` with `tool_choice` whenever the customer's message contains the word 'invoice' or 'charge'.

**Q8.** `apply_credit` returns the same tool result for every failure: `isError: true` with the text 'Operation failed'. The failures include billing-system timeouts, credits above the customer's monthly cap (a policy rule) and malformed account IDs. The agent retries cap violations again and again, and tells customers there is 'a technical problem'. What should the tool return instead?

- A) Raise an exception in the MCP server so the request fails at the protocol level, and let the session restart cleanly.
- B) Keep `isError: true` and add an error category, an `isRetryable` flag and an explanation the agent can relay.
- C) Return `isError: false` with the explanation in the text, so the agent treats the result as information rather than a failure.
- D) Leave the result as is, but wrap the tool in automatic retries with exponential backoff so transient failures resolve themselves.

**Q9.** `get_account` searches by phone number. It returns an empty list in two situations: when no account has that number, and when the CRM cannot be reached. In the second case the agent tells callers they have no account. What is the right fix?

- A) Return an empty success only when the search ran; return `isError: true` with a retryable transient error when the CRM is down.
- B) Have the agent retry every empty result twice before trusting it, which catches the cases where the CRM was briefly down.
- C) Escalate every empty search result to a human agent, who can check the CRM directly and confirm the caller's account.
- D) Keep the empty list in both cases, and add 'results may be incomplete when the CRM is slow or down' to the tool's description.

**Q10.** To save engineering time, the platform team exposed all 23 tools of the internal support toolkit to the agent, including bulk-admin and number-porting tools. Since then, tool-selection errors have tripled. What should you do?

- A) Set `tool_choice` to `"any"`, so the agent is always required to call one of the tools instead of guessing in text.
- B) Switch the agent to a larger model, which can reliably reason over a toolset of this size without extra configuration.
- C) Keep all 23 tools and add a paragraph to the system prompt that explains when each tool applies and when it does not.
- D) Give the agent only the few tools its role needs, and route rare admin actions through escalation or a specialized agent.

**Q11.** After each conversation a resolution record (category, credit amount, follow-up needed) must be written to the CRM. Today the system prompt says 'End with a JSON object', and about 3% of records fail to parse. You add a final Messages API call to produce the record, pinned to a model that still supports forced tool use. What is the most reliable design for that call?

- A) Repair malformed JSON with a tolerant parser and regex fixes before writing the record to the CRM.
- B) A `record_resolution` tool defined with `strict: true`, forced via `tool_choice: {"type": "tool", "name": "record_resolution"}`.
- C) Raise `max_tokens` for the final call, so the JSON object is never cut off before its closing brace.
- D) Add 'Respond only with valid JSON, with no prose and no markdown fences' in capitals to the prompt for that final call.

**Q12.** Even after the descriptions of `get_invoice` and `get_billing_history` were rewritten, the agent is inconsistent on vague requests such as 'my bill looks wrong' or 'why did I pay more this time?'. What is the best next step?

- A) Add 2–4 few-shot examples of these ambiguous requests, each showing the chosen tool and why it beats the alternative.
- B) Merge the two tools into one billing tool, so the agent never has to choose between invoice and history for vague requests.
- C) Set `tool_choice` to `"any"`, so that vague billing requests always lead to one of the two billing tools being called.
- D) Add about 20 examples of clear-cut requests, such as 'show my invoice' and 'list my payments', so tool use is demonstrated thoroughly.

**Q13.** A customer's first message is: 'I want to speak to a real person, not a bot.' The issue turns out to be a simple plan change that the agent could make in one step. What should the agent do?

- A) Make the plan change first, since it only takes one step, and then offer a human agent if the customer still wants one.
- B) Escalate to a human right away, with a handoff summary, without investigating or attempting the plan change first.
- C) Score the sentiment of the message and escalate only if it is strongly negative, since mild impatience is common.
- D) Ask why the customer prefers a human, so the agent can address the concern before deciding whether to escalate.

**Q14.** `get_account` called with the name 'J. Alvarez' returns three active accounts in the same city. What should the agent do?

- A) Escalate to a human, because acting on an uncertain identity is too risky for the agent to handle on its own.
- B) Ask the caller for an additional identifier, such as the account number or the phone number's last four digits.
- C) Read out the three accounts' details so the caller can confirm which one is theirs and the agent can continue.
- D) Pick the account with the most recent activity, since that is statistically the most likely to belong to the caller.

**Q15.** In long conversations, details drift after the history is summarized: the agent later mentions 'a small credit' when it had promised $35 on 12 September, and it loses the invoice number under discussion. What is the best fix?

- A) Summarize more frequently, so each summary covers fewer turns and has less room to lose the individual amounts and dates.
- B) Move to a model with a larger context window, so the history needs summarizing far less often in long conversations.
- C) Keep a structured case-facts block (amounts, dates, IDs, promises) and send it with every request, outside the summary.
- D) Have the agent ask the customer to restate the key details (amounts, dates, invoice numbers) whenever it is unsure.

---

## Scenario B: Code generation with Claude Code (Q16–Q30)

*Based on official scenario SC2.*

A 30-person product team uses **Claude Code** every day on a TypeScript monorepo (`apps/web`, `apps/api`, `packages/*`) plus a folder of Python data jobs. The repo has a project `CLAUDE.md`, a few `.claude/rules/` files and some shared skills. Engineers use Claude Code for code generation, refactoring, debugging and documentation. You own the team's Claude Code setup and its conventions.

**Q16.** A new engineer reports that Claude Code keeps running `npm install`, while everyone else's sessions use `pnpm` as the team requires. The `pnpm` rule turns out to live in each long-time engineer's `~/.claude/CLAUDE.md`. What is the right fix?

- A) Ask the new engineer to copy a senior engineer's `~/.claude/CLAUDE.md`, so their setup matches the rest of the team's.
- B) Run `/memory` on the new engineer's machine, so Claude Code syncs the memory files that teammates already have.
- C) Move the rule into the project `CLAUDE.md` (or `.claude/CLAUDE.md`), which is version-controlled and loads for everyone.
- D) Put the rule in `CLAUDE.local.md` at the repository root, where Claude Code picks it up for that project automatically.

**Q17.** A year ago the team wrote `.claude/rules/migrations.md` (reversible migrations, no data changes in schema migrations) with `paths: ["apps/api/db/migrations/*.sql"]`. Since then, new services keep migrations in their own folders, such as `packages/billing/migrations/2026/` and `apps/jobs/db/migrations/`. Claude follows the conventions in `apps/api` but ignores them everywhere else. The conventions should still load only when Claude works on a migration. What should you change?

- A) Delete the `paths` frontmatter, so the rule applies to every migration file in the repo, whichever service owns it.
- B) Create one rule file per service, each a copy of the conventions with that service's own migrations folder in `paths`.
- C) Ask engineers to add 'follow the migration conventions' to their prompt whenever they start writing a new migration.
- D) Widen the rule's `paths` to a pattern that matches migrations folders at any depth, such as `**/migrations/**/*.sql`.

**Q18.** Shared standards live in `standards/` (`typescript.md`, `react.md`, `python.md`, `api-errors.md`). Each package already has its own `CLAUDE.md`, and package maintainers know which standards apply to their package. They want those standards included without copying text. What should they do?

- A) Paste every standards file into the root `CLAUDE.md`, so all packages share one complete and consistent set of standards.
- B) In each package's `CLAUDE.md`, `@`-import the standards files that package needs, e.g. `@../../standards/react.md`.
- C) Turn each standards file into a skill with a clear description, and let Claude invoke the right one for each package.
- D) Copy the relevant standards text into each package's `CLAUDE.md`, so each package is self-contained and easy to read.

**Q19.** The team's `/map-dependencies` skill walks the whole repo: it greps and reads hundreds of files, producing about 2,000 lines of tool output, and ends with a one-page dependency summary. When engineers run it in the middle of a task, Claude afterwards loses track of what it was doing. What should you change?

- A) Move the skill's instructions into `CLAUDE.md`, so Claude always knows the dependency map without running anything.
- B) Ask engineers to run `/compact` after every run of the skill, so the 2,000 lines are summarized before work resumes.
- C) Add `context: fork` to the skill's frontmatter, so it runs in an isolated subagent context and returns only its result.
- D) Add `disable-model-invocation: true` to the frontmatter, so the skill only runs when an engineer explicitly asks for it.

**Q20.** An engineer wants a variant of the team's shared `/release-notes` skill that writes in a more casual style, for personal use only. Teammates must not be affected. What should they do?

- A) Add 'write release notes in a casual style' to the project `CLAUDE.md`, which the skill will pick up automatically.
- B) Create a personal skill under `~/.claude/skills/`, for example `~/.claude/skills/release-notes-casual/SKILL.md`.
- C) Add a `.claude/commands/release-notes-casual.md` command to the repository, next to the team's existing skill.
- D) Change the tone instructions in the repo's `.claude/skills/release-notes/SKILL.md` and keep the rest of the skill as it is.

**Q21.** *(Select TWO)* Which TWO of these tasks are best done with direct execution rather than plan mode?

- A) Add a maximum-length check to the `displayName` field validator.
- B) Extract the authentication code from the monolith into a separate service.
- C) Fix a null-reference error in one function; the stack trace points to the exact line.
- D) Replace the `moment` date library with `date-fns` across about 60 files.
- E) Choose between server-sent events and WebSockets for live notifications, given the current infrastructure.

**Q22.** Engineers ask Claude to convert legacy log lines into structured JSON events. The prose spec ('parse the timestamp, level and message; keep context fields') has produced three different output shapes in three runs. What is the most effective next step?

- A) Switch to plan mode before each conversion, so Claude designs the output format before writing it.
- B) Ask Claude to 'be consistent' and to reuse the output format it produced earlier in the session.
- C) Give 2–3 concrete examples of real input lines, each paired with the exact JSON event expected.
- D) Rewrite the spec in more detail, with a full paragraph for each field and its parsing rules.

**Q23.** Yesterday an engineer spent two hours in a named session, `payments-refactor`, analyzing about 40 files. Overnight a teammate merged changes to three of those files. Most of yesterday's analysis is still valid. How should the engineer continue?

- A) Fork the session and continue in the fork, because a fork re-reads any files that changed after the original session.
- B) Resume the session and carry on as before, because Claude re-reads files automatically when it needs them again.
- C) Start a new session and redo the full analysis, because any code change invalidates the earlier session completely.
- D) Resume with `claude --resume payments-refactor`, name the three changed files, and have Claude re-analyze just those.

**Q24.** After a long session that mapped the notification subsystem, an engineer wants to explore two refactoring approaches (an event bus vs. direct service calls) from that same analysis, without the two explorations contaminating each other. What should they use?

- A) Run `/compact` after exploring the first approach, then explore the second approach from the compacted summary.
- B) Fork the session once per approach (`--fork-session` with `--resume` in the CLI, `fork_session` in the SDK).
- C) Start two new sessions and repeat the subsystem mapping in each, so both explorations start from a clean context.
- D) Explore both approaches one after the other in the same session, so Claude can compare them directly at the end.

**Q25.** An engineer asks Claude to 'add comprehensive tests' to `packages/billing`, a 90-file legacy package with no tests and unclear dependencies. Which approach fits best?

- A) Map the dependencies first, find the riskiest areas, then follow a prioritized plan that adapts as you learn.
- B) Ask for all the tests in a single prompt, so Claude sees the whole package at once and can avoid duplicated tests.
- C) Start with the files that have the most lines of code, and stop when the token budget for the task runs out.
- D) Generate tests for every file in alphabetical order with one fixed per-file prompt, so coverage grows predictably.

**Q26.** Two hours into an investigation of a flaky integration test, Claude's answers become inconsistent and start referring to 'typical patterns in repos like this' instead of the specific classes it found earlier. What practice best counters this?

- A) Raise `max_tokens`, so Claude's answers can include the specific classes and the full detail of earlier findings.
- B) Switch to a model with a larger context window, so everything found earlier in the session stays in view for later answers.
- C) Retype the most important earlier findings from memory at the start of each new prompt, to keep them recent.
- D) Have Claude keep a scratchpad file of confirmed findings, with paths and evidence, and consult it for later questions.

**Q27.** Before migrating off a legacy HTTP client, Claude must find every call site across about 300 files. Earlier attempts filled the main session with search output, and the later implementation was inconsistent. What should you do?

- A) Paste the output of a repository-wide grep into the chat, so Claude gets every call site in one compact message.
- B) Delegate the discovery to a subagent, such as the built-in Explore subagent, which returns a summary of call sites.
- C) Run the discovery in the main session and run `/compact` whenever the context fills up with search output.
- D) Read all 300 files into the main session first, so Claude has the complete picture before it starts the migration.

**Q28.** *(Select TWO)* An engineer is about to run `/compact` in a long debugging session. The exact error codes and the reproduction steps must survive. Which TWO actions help most?

- A) Run `/compact keep the exact error codes and repro steps`, passing focus instructions for the summary.
- B) Rely on auto-compaction, which detects numbers and identifiers in the history and preserves them exactly.
- C) Run `/clear` instead of `/compact`, which frees the context window but keeps the full conversation history in view.
- D) Switch to plan mode before compacting, so the summary is written as a structured plan that keeps every detail.
- E) Write the exact error codes and repro steps to a notes file in the repo before compacting.

**Q29.** Every engineer should get the same GitHub MCP server in Claude Code, but the access token must never be committed. What configuration fits?

- A) Have each engineer add the server to their own `~/.claude.json`, which keeps every token on the engineer's machine.
- B) Commit the token in `.mcp.json`, since the repository is private and only employees can read its contents.
- C) Define the server in the project `.mcp.json` with `${GITHUB_TOKEN}` expansion; each engineer sets the token locally.
- D) Put the token in the project `CLAUDE.md`, so Claude can read it and pass it to the server when it connects.

**Q30.** *(Select TWO)* Which TWO pairings use the most appropriate built-in tool for the task?

- A) `Edit` with the pattern `**/*.ts` to rename a function in all files at once.
- B) `Glob` for `formatPrice` to find the files that call that function.
- C) `Read` every file under `src/`, one by one, to find where an error message is raised.
- D) `Grep` with the pattern `formatPrice\s*\(` to find call sites of that function.
- E) `Glob` with `**/*.spec.ts` to find spec files by name, wherever they live.

---

## Scenario C: Multi-agent research system (Q31–Q45)

*Based on official scenario SC3.*

An energy consultancy built a research system on the **Claude Agent SDK**. A **coordinator** delegates to four subagents: a **web-search** agent, a **document-analysis** agent (it works on an internal library of PDF reports exposed through an MCP server), a **synthesis** agent and a **report-writing** agent. Clients receive cited reports on questions such as *"What is the outlook for grid-scale battery storage in Southern Europe?"*

**Q31.** Every request, even 'When did Spain's 2023 storage auction close?', runs all four subagents in sequence and takes about 90 seconds. What change best fixes this?

- A) Cache the full pipeline's output, so repeated questions are answered instantly without running any subagents.
- B) Run all four subagents in parallel for every request, so total latency equals the slowest subagent rather than the sum.
- C) Remove the synthesis agent and let the report agent synthesize as well, which saves one hop on every request.
- D) Have the coordinator assess each query and invoke only the subagents it needs, e.g. only web search for a simple fact.

**Q32.** The coordinator gives two web-search subagents the same instruction: 'research battery storage in Southern Europe'. Their results overlap by about 70%. What is the best fix?

- A) Use a single web-search subagent for the whole topic, so there is nothing to duplicate in the first place.
- B) Give the two subagents different sampling temperatures, so their searches naturally diverge into different sources.
- C) Partition the scope when delegating: distinct subtopics or source types per subagent (e.g. policy vs. project costs).
- D) Keep the shared instruction and deduplicate both subagents' results in the synthesis step before writing the report.

**Q33.** The synthesis covers policy and project pipelines well but says almost nothing about grid-connection costs, which the client explicitly asked about. What should the system do?

- A) Ask the synthesis agent to fill the gap on grid-connection costs from its own general knowledge of the energy sector.
- B) Rerun the whole pipeline from scratch with the same instructions, since a fresh run usually covers all topics.
- C) Deliver the report on time and mention the missing topic in the cover email, offering a follow-up report later.
- D) Have the coordinator spot the gap, send targeted grid-cost queries to search and analysis, then re-run synthesis.

**Q34.** The synthesis subagent's reports are generic and ignore the document-analysis results. The coordinator's call to it reads: 'Synthesize the findings from the research phase into a report.' What is the root cause?

- A) The synthesis subagent does not inherit the coordinator's context; the findings must be passed in full in its prompt.
- B) Shared memory between subagents is not enabled, so the analysis results are not visible to the synthesis subagent.
- C) The synthesis subagent runs on a model that is too small to write a long report from many separate findings.
- D) The synthesis subagent lacks the document-analysis agent's MCP tools, so it cannot look at the analyzed documents itself.

**Q35.** Logs show the coordinator spawning three independent regional search subagents one per turn, waiting for each to finish. What change cuts latency the most?

- A) Merge the three regions into a single search subagent, which then covers all regions in one long run.
- B) Increase the coordinator's `max_turns`, so it can schedule all three regional subagents without ever hitting its turn limit.
- C) Have the coordinator emit all three subagent calls (`Agent`, formerly `Task`) in one response so they run concurrently.
- D) Have each regional search subagent spawn the next one when it finishes, so the coordinator no longer waits.

**Q36.** The coordinator's prompt gives each subagent a 20-step procedure ('search X, open the first three results, extract table 2 ...'). When sources are structured differently than expected, the subagents fail or return irrelevant data. What should change?

- A) Add automatic retries for the steps that fail, so a subagent can try the same procedure again on a different source.
- B) Add more steps to the procedure that cover each of the alternative source structures the subagents have run into so far.
- C) Delegate with the research goal and quality criteria (coverage, sourcing, dates), and let subagents adapt how they work.
- D) Replace the search subagents with a fixed scraping script, which follows the 20 steps exactly every single time.

**Q37.** To save time, a developer let the web-search subagent send its results straight to the report-writing subagent, bypassing the coordinator. Since then, failures on that path go unnoticed and reports sometimes miss the analysis results. What should you do?

- A) Let all subagents read and write a shared message bus, so any agent can pick up results the moment they are ready.
- B) Merge the search and report subagents into one agent, so results never have to travel between them at all.
- C) Add detailed logging inside each subagent, so that failures on the direct path become visible in the logs.
- D) Route all subagent communication back through the coordinator, which owns error handling, routing and observability.

**Q38.** The document-analysis subagent has a generic `fetch_url` tool. It sometimes fetches arbitrary web pages, duplicating the search agent's job and pulling in unvetted sources. What is the best change?

- A) Also give the analysis subagent the web-search tools, so that it at least fetches pages through the proper search path.
- B) Log every URL the subagent fetches and review the log weekly, removing unvetted sources from finished reports.
- C) Add a system-prompt rule for that subagent: 'Only fetch documents from the internal library, never other websites.'
- D) Replace `fetch_url` with a constrained `load_library_document(doc_id)` that only opens approved library documents.

**Q39.** One search provider returns intermittent HTTP 503 errors that usually clear within seconds. Currently the search subagent reports every 503 to the coordinator, which then re-delegates the whole search. What is the better design?

- A) Retry transient errors inside the search subagent (bounded) and escalate only if they persist, with attempts and partial results.
- B) Terminate the research run on the first 503, so the client never receives a report built on partial results.
- C) Have the search subagent return an empty result marked as successful when a 503 occurs, so the research run continues without delay.
- D) Keep reporting every 503 to the coordinator, but as a short generic 'search unavailable' status to save tokens.

**Q40.** Agents make dozens of exploratory `search_library` calls just to learn which reports exist in the internal library (titles, years, topics) before the real work starts. What is the best change?

- A) Add a `list_everything` tool that returns the full text of all documents, so one call replaces dozens of searches.
- B) Expose the library's catalog as an MCP resource, so agents can see what exists without exploratory tool calls.
- C) Raise `max_turns` for every agent, so that exploration no longer eats into the turns needed for the actual research.
- D) Put the full text of every library document into each agent's system prompt, so nothing needs to be searched.

**Q41.** The document-analysis subagent returns `null` for `methodology` on about 30% of papers. Review shows those papers do describe their method, just not under a 'Methods' heading: it is embedded in the results section or in a footnote. What fix is most effective?

- A) Post-process the papers with a regex that searches for a 'Methods' heading and passes that section along.
- B) Make `methodology` a required, non-nullable field in the schema, so the subagent can no longer return null.
- C) Add few-shot examples that extract the method from varied paper structures, including embedded and footnoted ones.
- D) Retry every extraction that returns a null `methodology`, up to three times, before accepting the empty value as final.

**Q42.** Document extraction runs as a direct Messages API call on a model that supports forced tool use. It offers three extraction tools (`extract_policy_doc`, `extract_market_report`, `extract_project_filing`) and does not know a document's type in advance. About 1 time in 20 Claude replies with prose instead of calling a tool. What setting fixes this?

- A) `tool_choice: {"type": "any"}`, so it must call one of the tools but can still choose which.
- B) `tool_choice: {"type": "tool", "name": "extract_market_report"}`, the most common document type.
- C) `tool_choice: {"type": "auto"}` with a firmer instruction to always use one of the extraction tools.
- D) `tool_choice: {"type": "none"}` with the JSON format described in the prompt instead.

**Q43.** For average 2024 battery pack prices, a bank's survey reports $139/kWh and an industry association reports $115/kWh. Both are credible and use different methods. The analysis subagent's prompt says 'resolve discrepancies', so it reports $127/kWh. What should it do instead?

- A) Stop the analysis and wait for a human analyst to choose a value before continuing with the remaining documents.
- B) Report the figure from the source with the larger sample, since a larger sample gives the more reliable estimate.
- C) Report the lower figure, since a lower cost estimate is the more conservative choice for clients making investment decisions.
- D) Report both values with sources, dates and methods, flag the conflict, and let the coordinator decide how to present it.

**Q44.** The report flags a 'contradiction': one source says 32% of planned solar projects included storage, another says 58%. The first figure comes from a 2021 survey and the second from 2025. What change prevents this misreading?

- A) Leave percentages out of the report whenever two sources disagree, and describe the trend qualitatively instead.
- B) Average the two figures and report the range, so the report reflects both sources without taking a side.
- C) Instruct the synthesis agent to always prefer the most recent source and drop older figures on the same metric.
- D) Require publication and data-collection dates in subagent outputs, so synthesis reads the gap as change over time.

**Q45.** *(Select TWO)* The web-search subagent for 'permitting timelines' failed even after local retries; every other subtopic succeeded. Which TWO actions belong in the reporting flow?

- A) Leave the permitting topic out of the report without comment, since the remaining sections are complete and sourced.
- B) Hold the entire report until the search source recovers, so the client receives one complete report, not a partial one.
- C) Proceed with the other results and mark the permitting section as a coverage gap (sources unavailable).
- D) State in the permitting section that no evidence was found on permitting timelines, so readers know it was researched.
- E) Separate well-supported findings from areas with gaps or single sources in the synthesis.

---

## Scenario D: Claude Code for continuous integration (Q46–Q60)

*Based on official scenario SC5.*

A payments company runs **Claude Code in GitHub Actions**. On every pull request it reviews the diff and posts inline comments, and it generates unit tests for changed code. Nightly and weekly jobs produce audits and reports, and a small pre-review step is written directly against the **Messages API**. Developers have started ignoring the review comments because many of them are noise. You own the pipeline, its prompts and its configuration.

**Q46.** Each finding should become an inline PR comment at the right file and line. Today the job prints a markdown review that a script parses with regular expressions, and it breaks every few days. What should you change?

- A) Run with `--output-format json` plus a `--json-schema` for findings, then read the `structured_output` field.
- B) Add 'format every finding as FILE:LINE: message' to the prompt and keep the regex parser for posting the comments.
- C) Write the exact expected output format into the project `CLAUDE.md`, which every CI run of Claude Code loads.
- D) Use `--output-format stream-json` and parse each streamed text chunk as it arrives, to catch format errors earlier.

**Q47.** *(Select TWO)* Generated unit tests often ignore the team's fixture factories and duplicate scenarios that existing tests already cover. Which TWO changes help most?

- A) Document the testing standards, valuable-test criteria and fixtures in the project `CLAUDE.md`, which CI runs read.
- B) Run the job with `--bare`, so that personal settings and local configuration on the runner cannot interfere.
- C) Move test generation to the Message Batches API, so each changed module gets a separate, more focused request.
- D) Raise `--max-turns` so Claude has more turns to discover the fixture factories and existing tests by itself.
- E) Include the module's existing test files in the context of the generation step.

**Q48.** When new commits are pushed, the review re-runs and posts the same nine comments again, including ones the developer already fixed. What is the best fix?

- A) Delete all earlier bot comments before each run, then post the full new review, so only current comments remain.
- B) Review only the diff of the latest commit, since the earlier commits were already reviewed in previous runs.
- C) Pass the previous review's findings into the new run and ask for only new issues and issues still unaddressed.
- D) Drop new comments whose text exactly matches an earlier comment on the same PR, before posting the review.

**Q49.** A job generates a fix and then asks the same Claude Code session to review that fix before opening the PR. The review almost never finds problems, but human reviewers often do. What is the best change?

- A) Have the same session review the fix twice in a row, and open the PR only if both passes find no problems.
- B) Enable extended thinking for the review step in the same session, so it reasons more deeply about the fix.
- C) Run the review in a separate, independent instance that gets the diff but not the generating session's reasoning.
- D) Add 'be highly critical of your own code and assume it contains bugs' to the review prompt used in the same session.

**Q50.** *(Select TWO)* Which TWO statements about running Claude Code in a CI job are correct?

- A) CI jobs must call the Message Batches API instead, because Claude Code cannot run without a terminal.
- B) `claude -p "<prompt>"` runs non-interactively: it processes the prompt, prints the result and exits.
- C) With `--output-format json`, the output includes the result plus metadata such as the session ID and cost.
- D) Claude Code skips the project `CLAUDE.md` whenever it runs with `-p`, so CI guidance must go in the prompt.
- E) `--json-schema` works only in interactive sessions, where Claude can ask the user to fix invalid output.

**Q51.** The review prompt says 'Check that code comments are accurate.' Most resulting comments are speculative ('this comment may be outdated'), and developers dismiss them. Which prompt change best improves precision?

- A) Only report comment issues you are highly confident about, and skip anything you are unsure of.
- B) Flag a comment only when the behavior it claims clearly contradicts what the code does, and quote both.
- C) Be conservative when flagging comment issues, since developers find speculative comments annoying.
- D) Report at most three comment issues per PR, so the important ones are not buried under minor ones.

**Q52.** Style findings are dismissed 80% of the time, and developers have started dismissing the accurate security findings as well. What should you do first?

- A) Require developers to reply to every finding before they can merge, so each dismissal is a deliberate decision.
- B) Switch the review to a larger model, which makes fewer false-positive style findings in the first place.
- C) Keep all categories but post only findings whose self-reported confidence is above 0.9, to cut the noise across the board.
- D) Temporarily disable the style category to restore trust in the other findings, while improving its criteria offline.

**Q53.** The reviewer keeps flagging the team's deliberate `// eslint-disable-next-line` lines, each of which carries a justification comment, while it correctly flags disables that have no justification. What is the most effective fix?

- A) Remove the lint category from the review entirely, since the lint tooling already enforces its own rules.
- B) Add 'do not report false positives about lint suppressions' to the review prompt, near the review criteria.
- C) Add few-shot examples contrasting a justified disable with a genuine problem, explaining the difference.
- D) Filter out every finding whose text mentions 'eslint' in post-processing, before the comments are posted.

**Q54.** You want to understand systematically why developers dismiss certain findings, so you can fix the prompts that cause them. What should you add?

- A) A request that developers rate each posted comment from 1 to 5, so the team can see which kinds of comments are least useful.
- B) A higher severity threshold for posting, so fewer low-value findings reach developers in the first place.
- C) A `detected_pattern` field on each finding (the construct that triggered it), so dismissals can be grouped by pattern.
- D) A free-text reasoning paragraph on each finding, so reviewers can read exactly why it was raised.

**Q55.** Every merged PR must get an architecture-impact summary within 36 hours of the merge. Summaries are produced through the Message Batches API. You plan for the worst case of 24 hours of batch processing and reserve 4 hours after each batch to validate results and post the summaries. For this calculation, assume every request finishes within the 24-hour window. What is the longest submission interval for accumulated merges that still meets the requirement?

- A) Every 8 hours.
- B) Every 16 hours, because most batches finish well within 24 hours.
- C) Every 12 hours.
- D) Every 24 hours.

**Q56.** *(Select TWO)* Last night's batch of 600 file-analysis requests ended with 588 `succeeded` and 12 `errored` results. The errors show that those 12 files exceeded the context window. Which TWO actions are correct?

- A) Move all 600 files to synchronous calls from now on.
- B) Resubmit only the 12 failed requests, identified by their `custom_id`.
- C) Resubmit the 12 requests unchanged, since batch errors are usually transient.
- D) Resubmit the full batch of 600 so all results come from the same run.
- E) Split the oversized files into smaller chunks before resubmitting them.

**Q57.** The pipeline already reviews each changed file in its own pass. A recent PR changed a function in one file to return `null` on failure instead of throwing, and a caller in another file was not updated. No pass caught it. What should you add?

- A) Raise the output-token limit for each per-file pass, so the reviewer has room to describe effects on other files.
- B) Merge all changed files back into a single review pass, so the reviewer sees every file and its callers together.
- C) A separate integration pass on cross-file data flow, fed the changed interfaces and the per-file findings.
- D) Run each per-file pass several times and post only the findings that a majority of the runs agree on.

**Q58.** You want confident findings to go straight to PR comments and uncertain ones to a human triage queue. Which design is sound?

- A) Have the model give a confidence per finding, calibrate thresholds on labeled past findings, and triage those below.
- B) Ask the model to decide for each finding whether a human should see it, and follow its decision.
- C) Post every finding whose self-reported confidence is above 0.9, and send the rest to the triage queue.
- D) Route findings by the length of their explanation, since long, hedged explanations usually signal an uncertain finding.

**Q59.** In the pre-review step, written against the Messages API on a model that supports forced tool use, Claude must call `classify_pr_risk` before any review tool, so later steps can pick the review depth. Sometimes it starts reviewing first. What should you configure?

- A) Force the first request with `tool_choice: {"type": "tool", "name": "classify_pr_risk"}`; use `auto` after that.
- B) List `classify_pr_risk` first in the `tools` array, since Claude considers the tools in the order they are defined.
- C) Add 'always call classify_pr_risk before any other tool' to the system prompt, near the top.
- D) Set `tool_choice: {"type": "any"}` on every request in the step, so that Claude always starts with a tool call.

**Q60.** The review should check each PR against the acceptance criteria in its linked Jira ticket. A developer proposes writing a custom Jira MCP server. What do you recommend?

- A) Write the custom server, since owning the code gives the team full control over security, data handling and future features.
- B) Use an existing, maintained Jira MCP server for this standard integration; keep custom servers for team-specific needs.
- C) Give Claude Code a generic HTTP request tool and the Jira API key, so it can query any Jira endpoint it needs.
- D) Paste the linked ticket's text into the project `CLAUDE.md` for each PR, so the review always sees the criteria.

---

## Your answers

Fill in every item, then run the grader. Unanswered items count as wrong, as on the real exam.

In [ ]:
answers = {
    # Scenario A: Customer support resolution agent
    "q1": None,
    "q2": None,  # Select TWO
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
    "q11": None,
    "q12": None,
    "q13": None,
    "q14": None,
    "q15": None,
    # Scenario B: Code generation with Claude Code
    "q16": None,
    "q17": None,
    "q18": None,
    "q19": None,
    "q20": None,
    "q21": None,  # Select TWO
    "q22": None,
    "q23": None,
    "q24": None,
    "q25": None,
    "q26": None,
    "q27": None,
    "q28": None,  # Select TWO
    "q29": None,
    "q30": None,  # Select TWO
    # Scenario C: Multi-agent research system
    "q31": None,
    "q32": None,
    "q33": None,
    "q34": None,
    "q35": None,
    "q36": None,
    "q37": None,
    "q38": None,
    "q39": None,
    "q40": None,
    "q41": None,
    "q42": None,
    "q43": None,
    "q44": None,
    "q45": None,  # Select TWO
    # Scenario D: Claude Code for continuous integration
    "q46": None,
    "q47": None,  # Select TWO
    "q48": None,
    "q49": None,
    "q50": None,  # Select TWO
    "q51": None,
    "q52": None,
    "q53": None,
    "q54": None,
    "q55": None,
    "q56": None,  # Select TWO
    "q57": None,
    "q58": None,
    "q59": None,
    "q60": None,
}

## Grader and score report

The report mirrors the real one: percent correct by domain, a total and a scaled score, plus (because this is a study tool) the task statements and sections behind each miss.

**Scaled-score approximation (not official).** A piecewise-linear map through three points: 0% raw → 100, 72% raw → 720, 100% raw → 1,000. It keeps this repo's 72% study bar aligned with the 720 cut score. The real exam uses a scaling model from a standard-setting study that also equates forms of different difficulty; its conversion is not published.

In [ ]:
DOMAINS = {1: 'Agentic Architecture & Orchestration', 2: 'Tool Design & MCP Integration', 3: 'Claude Code Configuration & Workflows', 4: 'Prompt Engineering & Structured Output', 5: 'Context Management & Reliability'}
WEIGHTS = {1: 27, 2: 18, 3: 20, 4: 20, 5: 15}
META = {
 "q1": {
  "scenario": "A",
  "domain": 1,
  "task": "1.5",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.5",
    "../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk"
   ],
   [
    "Ch 9 §9.4",
    "../../09-escalation-human-in-the-loop/README.md#94-programmatic-enforcement-vs-prompt-guidance"
   ]
  ]
 },
 "q2": {
  "scenario": "A",
  "domain": 1,
  "task": "1.4",
  "n": 2,
  "review": [
   [
    "Ch 9 §9.4",
    "../../09-escalation-human-in-the-loop/README.md#94-programmatic-enforcement-vs-prompt-guidance"
   ],
   [
    "Ch 3 §3.5",
    "../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk"
   ]
  ]
 },
 "q3": {
  "scenario": "A",
  "domain": 1,
  "task": "1.1",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.1",
    "../../03-agent-sdk/README.md#31-the-agentic-loop"
   ],
   [
    "Ch 1 §1.3",
    "../../01-claude-api-fundamentals/README.md#13-stop_reason-why-the-model-stopped"
   ],
   [
    "Ch 2 §2.6",
    "../../02-tool-use/README.md#26-the-client-side-tool-loop-and-tool_result-rules-expanded-beyond-the-guide"
   ]
  ]
 },
 "q4": {
  "scenario": "A",
  "domain": 1,
  "task": "1.4",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.6",
    "../../09-escalation-human-in-the-loop/README.md#96-structured-handoff-protocols"
   ]
  ]
 },
 "q5": {
  "scenario": "A",
  "domain": 1,
  "task": "1.4",
  "n": 1,
  "review": [
   [
    "Ch 8 §8.9",
    "../../08-task-decomposition/README.md#89-decomposing-multi-concern-requests"
   ],
   [
    "Ch 9 §9.6",
    "../../09-escalation-human-in-the-loop/README.md#96-structured-handoff-protocols"
   ]
  ]
 },
 "q6": {
  "scenario": "A",
  "domain": 1,
  "task": "1.5",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.5",
    "../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk"
   ]
  ]
 },
 "q7": {
  "scenario": "A",
  "domain": 2,
  "task": "2.1",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.2",
    "../../02-tool-use/README.md#22-tool-definition"
   ]
  ]
 },
 "q8": {
  "scenario": "A",
  "domain": 2,
  "task": "2.2",
  "n": 1,
  "review": [
   [
    "Ch 10 §10.4",
    "../../10-multi-agent-error-handling/README.md#104-structured-error-responses-from-tools"
   ],
   [
    "Ch 10 §10.2",
    "../../10-multi-agent-error-handling/README.md#102-error-categories-and-the-retryable-flag"
   ],
   [
    "Ch 4 §4.4",
    "../../04-model-context-protocol/README.md#44-the-iserror-flag-and-structured-errors"
   ]
  ]
 },
 "q9": {
  "scenario": "A",
  "domain": 2,
  "task": "2.2",
  "n": 1,
  "review": [
   [
    "Ch 10 §10.3",
    "../../10-multi-agent-error-handling/README.md#103-access-failure-vs-valid-empty-result"
   ]
  ]
 },
 "q10": {
  "scenario": "A",
  "domain": 2,
  "task": "2.3",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.2",
    "../../02-tool-use/README.md#22-tool-definition"
   ],
   [
    "Ch 3 §3.2",
    "../../03-agent-sdk/README.md#32-agentdefinition-configuration"
   ]
  ]
 },
 "q11": {
  "scenario": "A",
  "domain": 4,
  "task": "4.3",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.3",
    "../../02-tool-use/README.md#23-the-tool_choice-parameter"
   ],
   [
    "Ch 2 §2.4",
    "../../02-tool-use/README.md#24-json-schemas-for-structured-output"
   ]
  ]
 },
 "q12": {
  "scenario": "A",
  "domain": 4,
  "task": "4.2",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.1",
    "../../06-prompt-engineering/README.md#61-few-shot-prompting"
   ],
   [
    "Ch 2 §2.2",
    "../../02-tool-use/README.md#22-tool-definition"
   ]
  ]
 },
 "q13": {
  "scenario": "A",
  "domain": 5,
  "task": "5.2",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.2",
    "../../09-escalation-human-in-the-loop/README.md#92-escalation-patterns"
   ],
   [
    "Ch 9 §9.1",
    "../../09-escalation-human-in-the-loop/README.md#91-when-to-escalate"
   ]
  ]
 },
 "q14": {
  "scenario": "A",
  "domain": 5,
  "task": "5.2",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.3",
    "../../09-escalation-human-in-the-loop/README.md#93-resolving-ambiguity-ask-dont-guess"
   ]
  ]
 },
 "q15": {
  "scenario": "A",
  "domain": 5,
  "task": "5.1",
  "n": 1,
  "review": [
   [
    "Ch 11 §11.1",
    "../../11-context-management/README.md#111-extract-facts-into-a-persistent-case-facts-block"
   ]
  ]
 },
 "q16": {
  "scenario": "B",
  "domain": 3,
  "task": "3.1",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.1",
    "../../05-claude-code/README.md#51-the-claudemd-hierarchy"
   ]
  ]
 },
 "q17": {
  "scenario": "B",
  "domain": 3,
  "task": "3.3",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.3",
    "../../05-claude-code/README.md#53-clauderules-topic-files-and-path-scoped-rules"
   ]
  ]
 },
 "q18": {
  "scenario": "B",
  "domain": 3,
  "task": "3.1",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.2",
    "../../05-claude-code/README.md#52-path-imports-modular-claudemd"
   ]
  ]
 },
 "q19": {
  "scenario": "B",
  "domain": 3,
  "task": "3.2",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.5",
    "../../05-claude-code/README.md#55-skills-in-depth-skillmd-frontmatter"
   ],
   [
    "Ch 11 §11.5",
    "../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context"
   ]
  ]
 },
 "q20": {
  "scenario": "B",
  "domain": 3,
  "task": "3.2",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.4",
    "../../05-claude-code/README.md#54-custom-slash-commands--skills"
   ]
  ]
 },
 "q21": {
  "scenario": "B",
  "domain": 3,
  "task": "3.4",
  "n": 2,
  "review": [
   [
    "Ch 5 §5.6",
    "../../05-claude-code/README.md#56-plan-mode-vs-direct-execution"
   ]
  ]
 },
 "q22": {
  "scenario": "B",
  "domain": 3,
  "task": "3.5",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.9",
    "../../06-prompt-engineering/README.md#69-matching-the-strategy-to-the-task-and-iterating-expanded-beyond-the-guide"
   ],
   [
    "Ch 6 §6.1",
    "../../06-prompt-engineering/README.md#61-few-shot-prompting"
   ]
  ]
 },
 "q23": {
  "scenario": "B",
  "domain": 1,
  "task": "1.7",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.10",
    "../../05-claude-code/README.md#510-sessions-resume-name-fork"
   ],
   [
    "Ch 11 §11.7",
    "../../11-context-management/README.md#117-sessions-resume-fork-or-start-fresh-task-17"
   ]
  ]
 },
 "q24": {
  "scenario": "B",
  "domain": 1,
  "task": "1.7",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.10",
    "../../05-claude-code/README.md#510-sessions-resume-name-fork"
   ],
   [
    "Ch 3 §3.6",
    "../../03-agent-sdk/README.md#36-sessions-resume-and-fork-bonus-from-domain-17"
   ]
  ]
 },
 "q25": {
  "scenario": "B",
  "domain": 1,
  "task": "1.6",
  "n": 1,
  "review": [
   [
    "Ch 8 §8.2",
    "../../08-task-decomposition/README.md#82-dynamic-adaptive-decomposition"
   ]
  ]
 },
 "q26": {
  "scenario": "B",
  "domain": 5,
  "task": "5.4",
  "n": 1,
  "review": [
   [
    "Ch 11 §11.4",
    "../../11-context-management/README.md#114-scratchpad-files"
   ]
  ]
 },
 "q27": {
  "scenario": "B",
  "domain": 5,
  "task": "5.4",
  "n": 1,
  "review": [
   [
    "Ch 11 §11.5",
    "../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context"
   ],
   [
    "Ch 5 §5.6",
    "../../05-claude-code/README.md#56-plan-mode-vs-direct-execution"
   ]
  ]
 },
 "q28": {
  "scenario": "B",
  "domain": 5,
  "task": "5.4",
  "n": 2,
  "review": [
   [
    "Ch 5 §5.7",
    "../../05-claude-code/README.md#57-compact-and-managing-the-context-window"
   ],
   [
    "Ch 11 §11.4",
    "../../11-context-management/README.md#114-scratchpad-files"
   ]
  ]
 },
 "q29": {
  "scenario": "B",
  "domain": 2,
  "task": "2.4",
  "n": 1,
  "review": [
   [
    "Ch 4 §4.3",
    "../../04-model-context-protocol/README.md#43-configuring-mcp-servers"
   ]
  ]
 },
 "q30": {
  "scenario": "B",
  "domain": 2,
  "task": "2.5",
  "n": 2,
  "review": [
   [
    "Ch 13 §13.2",
    "../../13-claude-code-builtin-tools/README.md#132-grep-vs-glob-content-search-vs-path-search"
   ],
   [
    "Ch 13 §13.3",
    "../../13-claude-code-builtin-tools/README.md#133-read-write-and-edit-full-file-vs-targeted-changes"
   ]
  ]
 },
 "q31": {
  "scenario": "C",
  "domain": 1,
  "task": "1.2",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.3",
    "../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents"
   ],
   [
    "Ch 8 §8.7",
    "../../08-task-decomposition/README.md#87-workflow-or-agent-choosing-the-level-of-autonomy"
   ]
  ]
 },
 "q32": {
  "scenario": "C",
  "domain": 1,
  "task": "1.2",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.3",
    "../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents"
   ]
  ]
 },
 "q33": {
  "scenario": "C",
  "domain": 1,
  "task": "1.2",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.3",
    "../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents"
   ],
   [
    "Ch 8 §8.2",
    "../../08-task-decomposition/README.md#82-dynamic-adaptive-decomposition"
   ]
  ]
 },
 "q34": {
  "scenario": "C",
  "domain": 1,
  "task": "1.3",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.4",
    "../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task"
   ],
   [
    "Ch 11 §11.5",
    "../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context"
   ]
  ]
 },
 "q35": {
  "scenario": "C",
  "domain": 1,
  "task": "1.3",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.4",
    "../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task"
   ]
  ]
 },
 "q36": {
  "scenario": "C",
  "domain": 1,
  "task": "1.3",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.3",
    "../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents"
   ],
   [
    "Ch 3 §3.4",
    "../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task"
   ]
  ]
 },
 "q37": {
  "scenario": "C",
  "domain": 1,
  "task": "1.2",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.3",
    "../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents"
   ],
   [
    "Ch 10 §10.1",
    "../../10-multi-agent-error-handling/README.md#101-where-failures-happen-in-a-multi-agent-system-expanded-beyond-the-guide"
   ]
  ]
 },
 "q38": {
  "scenario": "C",
  "domain": 2,
  "task": "2.3",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.2",
    "../../02-tool-use/README.md#22-tool-definition"
   ],
   [
    "Ch 3 §3.2",
    "../../03-agent-sdk/README.md#32-agentdefinition-configuration"
   ]
  ]
 },
 "q39": {
  "scenario": "C",
  "domain": 2,
  "task": "2.2",
  "n": 1,
  "review": [
   [
    "Ch 10 §10.5",
    "../../10-multi-agent-error-handling/README.md#105-local-recovery-before-propagation"
   ]
  ]
 },
 "q40": {
  "scenario": "C",
  "domain": 2,
  "task": "2.4",
  "n": 1,
  "review": [
   [
    "Ch 4 §4.5",
    "../../04-model-context-protocol/README.md#45-resources-give-the-agent-a-map"
   ]
  ]
 },
 "q41": {
  "scenario": "C",
  "domain": 4,
  "task": "4.2",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.1",
    "../../06-prompt-engineering/README.md#61-few-shot-prompting"
   ]
  ]
 },
 "q42": {
  "scenario": "C",
  "domain": 4,
  "task": "4.3",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.3",
    "../../02-tool-use/README.md#23-the-tool_choice-parameter"
   ]
  ]
 },
 "q43": {
  "scenario": "C",
  "domain": 5,
  "task": "5.6",
  "n": 1,
  "review": [
   [
    "Ch 12 §12.2",
    "../../12-provenance/README.md#122-handling-conflicting-data"
   ]
  ]
 },
 "q44": {
  "scenario": "C",
  "domain": 5,
  "task": "5.6",
  "n": 1,
  "review": [
   [
    "Ch 12 §12.3",
    "../../12-provenance/README.md#123-include-dates-for-correct-interpretation"
   ]
  ]
 },
 "q45": {
  "scenario": "C",
  "domain": 5,
  "task": "5.3",
  "n": 2,
  "review": [
   [
    "Ch 10 §10.9",
    "../../10-multi-agent-error-handling/README.md#109-coverage-annotations-in-the-final-synthesis"
   ],
   [
    "Ch 12 §12.6",
    "../../12-provenance/README.md#126-coverage-and-uncertainty-reporting-expanded-beyond-the-guide"
   ],
   [
    "Ch 10 §10.3",
    "../../10-multi-agent-error-handling/README.md#103-access-failure-vs-valid-empty-result"
   ]
  ]
 },
 "q46": {
  "scenario": "D",
  "domain": 3,
  "task": "3.6",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.9",
    "../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode"
   ]
  ]
 },
 "q47": {
  "scenario": "D",
  "domain": 3,
  "task": "3.6",
  "n": 2,
  "review": [
   [
    "Ch 5 §5.9",
    "../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode"
   ]
  ]
 },
 "q48": {
  "scenario": "D",
  "domain": 3,
  "task": "3.6",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.9",
    "../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode"
   ]
  ]
 },
 "q49": {
  "scenario": "D",
  "domain": 3,
  "task": "3.6",
  "n": 1,
  "review": [
   [
    "Ch 8 §8.4",
    "../../08-task-decomposition/README.md#84-independent-review-instances-why-self-review-is-weak"
   ],
   [
    "Ch 5 §5.9",
    "../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode"
   ]
  ]
 },
 "q50": {
  "scenario": "D",
  "domain": 3,
  "task": "3.6",
  "n": 2,
  "review": [
   [
    "Ch 5 §5.9",
    "../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode"
   ]
  ]
 },
 "q51": {
  "scenario": "D",
  "domain": 4,
  "task": "4.1",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.2",
    "../../06-prompt-engineering/README.md#62-explicit-criteria-vs-vague-instructions"
   ]
  ]
 },
 "q52": {
  "scenario": "D",
  "domain": 4,
  "task": "4.1",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.2",
    "../../06-prompt-engineering/README.md#62-explicit-criteria-vs-vague-instructions"
   ]
  ]
 },
 "q53": {
  "scenario": "D",
  "domain": 4,
  "task": "4.2",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.1",
    "../../06-prompt-engineering/README.md#61-few-shot-prompting"
   ]
  ]
 },
 "q54": {
  "scenario": "D",
  "domain": 4,
  "task": "4.4",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.6",
    "../../06-prompt-engineering/README.md#66-self-correction"
   ]
  ]
 },
 "q55": {
  "scenario": "D",
  "domain": 4,
  "task": "4.5",
  "n": 1,
  "review": [
   [
    "Ch 7 §7.8",
    "../../07-message-batches/README.md#78-sla-planning-and-submission-cadence"
   ]
  ]
 },
 "q56": {
  "scenario": "D",
  "domain": 4,
  "task": "4.5",
  "n": 2,
  "review": [
   [
    "Ch 7 §7.6",
    "../../07-message-batches/README.md#76-result-types-and-resubmitting-only-the-failures"
   ]
  ]
 },
 "q57": {
  "scenario": "D",
  "domain": 4,
  "task": "4.6",
  "n": 1,
  "review": [
   [
    "Ch 8 §8.3",
    "../../08-task-decomposition/README.md#83-multi-pass-code-review"
   ]
  ]
 },
 "q58": {
  "scenario": "D",
  "domain": 4,
  "task": "4.6",
  "n": 1,
  "review": [
   [
    "Ch 8 §8.5",
    "../../08-task-decomposition/README.md#85-verification-passes-and-confidence-based-routing"
   ],
   [
    "Ch 9 §9.7",
    "../../09-escalation-human-in-the-loop/README.md#97-confidence-calibration"
   ]
  ]
 },
 "q59": {
  "scenario": "D",
  "domain": 2,
  "task": "2.3",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.3",
    "../../02-tool-use/README.md#23-the-tool_choice-parameter"
   ]
  ]
 },
 "q60": {
  "scenario": "D",
  "domain": 2,
  "task": "2.4",
  "n": 1,
  "review": [
   [
    "Ch 4: Architect decision cheat-sheet",
    "../../04-model-context-protocol/README.md#architect-decision-cheat-sheet"
   ]
  ]
 }
}
KEY_HASH = {
 "q1": "d01e0d2123c2608d1604388f402be6311fec6f81aacc56159f9974fc375704dd",
 "q2": "2855c7aec70fd793023f3646477ffe9a45c26f19da3594520814dad06651f34f",
 "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
 "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
 "q5": "efeafc86344da0b52bbaa7783b6621595e70274062976345ce6c533b9322877c",
 "q6": "ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522",
 "q7": "16dd052a869d89a87df9ae09526a5477a3e6870575379b0c9d2ed35de3ab21c7",
 "q8": "39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834",
 "q9": "8b77cfcdcf7bee660b161787fd482af13fe960d0da61692597e6e870d4668936",
 "q10": "c72adbd855b96e33ffae9308bfc1d5cdfa7fb315301530c3b1f890435529ec04",
 "q11": "42310be80adadfed411c1783b5c42432624ffc39d2ea2f7d837996726d6b0774",
 "q12": "9d4e72c9d2036d777ce76bca900a85c706c8a653c712a6faf259d63490f850a2",
 "q13": "1f4d997a7129bb42fd213d163d4614aaf9eb4f263d7a3fa20c4fb71f1117d690",
 "q14": "3aad1d5caed969668cf4a743f4973dd35d6b6c8c1038601173f296d405aa5e7e",
 "q15": "ae48acbfadb880d467172beaefcdb7dbea08f91ef4ec6009be505058f35e4ca0",
 "q16": "5ed42e68e28f04f3dc09591f0b67624b3ff8ee65eaec73eb88952270fe5b63c6",
 "q17": "10b258f19dc0d836f831aaf923eb91cb349f54e1914a54f586b8e54469bd8b4f",
 "q18": "c18505579193f4c6c2c72adcafbfdafb1b0dc54f4d88850311c7615b46ce9183",
 "q19": "1f53d76ad7625e5655c12583514b1f5b7c731e828c70a5c8b8ac0ff467c7c036",
 "q20": "2effc2eaafea93bcef1e4ab2575ec36d71459e03adf49cab59176d5f45f3421d",
 "q21": "a57aba2cff8fcd30826e7c1c745c835caa826daeb39e83fe10a33f66f35e9955",
 "q22": "9b08b9f5382cf9e463a474564b2958d1ddb11cdf7a0b12e7b9d40c27c8e99998",
 "q23": "bcd6cfb076bbc0b6f7e9b6a919fb987e48f41b58a2ae7f1ea439f26e0bde64a2",
 "q24": "93a0a4e5f03f9f3a7daf7a64f1975fc28c32676f6d52bbd3b2f699418d743103",
 "q25": "319ea34456d79f6a3448f520aec57829e31bb551cc5173c92633c36f66d160e0",
 "q26": "883736b0f1430070851c7473d136972a6ed9b74ba90f827b566dd492d4bb90de",
 "q27": "5a7c9285166ceb7a48c537cc8d76079eddf1560194a293a3edf193ae3826f9d9",
 "q28": "3134ca6f3754a7122a5f8a7ebb81b5ea7b6999148c7579d6b7cb563c3d165832",
 "q29": "f4ea735c1d64f0352f59ab6c3b63f672f6c4b8b1ae6082127cb07dcf291e542a",
 "q30": "8d284e516f40e53594ae13bd855742e0eb84960968c8af9b308a184b80bbec29",
 "q31": "886942bb2abcffc22325961fc11617e90dbb333e01fd66e9daa61fd011d51c02",
 "q32": "57663978fdb314f4cc5975ef656cd962af735d9f497fa22fc2f46127a0fc88d2",
 "q33": "0ec5c6bc8571ce076f344c911bfcf6c139b407074e778f60fb7e7e2ca6a7f9c8",
 "q34": "8c073cc57bc34ba3613d95acc523318934fd67c63fca5aefaede75f191441c5b",
 "q35": "24d0a6bada560b1961ea2e194250b6ac9e0e7d0c72e0d824b09bb505d9824864",
 "q36": "5930b2f687e64140f2171e773bb1f8b874fb2847c5740f769922f4c7afbd2103",
 "q37": "be06b5c9db773879ab6fc10f2421dfd189627042136917d5c33e8780089b16d6",
 "q38": "4fc0560baf15283da24cf0e8b62d47e5d3639921999038459c96e469dcacec4c",
 "q39": "fdb74eb3fe7cb05e94a5af76a675c6c77975a2042c04523e7565a60e364d244a",
 "q40": "55c866c0e5b83984a8abefb572f70805443ae26e16d75cc8edc29ef5648b7305",
 "q41": "62577fc5b2914e492f2d7cf0bf31064df2536b5cce8cebfc20efe739a68692d5",
 "q42": "198aea7ea5599afaa5b2debae24810e562f8d01cc218e10afc08a0226008f504",
 "q43": "6a3fd442ce5e944ffad0ee1a45e162a910eb014f9afe9ba38f848ea6e31a67e6",
 "q44": "e5de764906328084918aea63bc21f066850d1e01f178b981d6e1bf773cf5ad9f",
 "q45": "017daafb19e3be67a8b1e3497010d196c5adbb7956eecd13baecaeaba8e268b7",
 "q46": "61f70c4975ad50601e8f8e07a96286f528831d52d8583aeae8615a63241f5f92",
 "q47": "7c29afdaefc0c279a1ac51b46df5ee8b9727ae26b09e50853444ff205315d98b",
 "q48": "eda83dd219b11af8bf9436144eabff10f1ae1c60f61e6d011da882db5136a8f2",
 "q49": "e3a87eb2e0af9401b9dbd57747b88d6537106b8e74394269a4e6e28af34b410d",
 "q50": "cd2ace62a21a05dd7ae95e23ed429df5a836341650359ec6b8f2e7ee843f4351",
 "q51": "92126b77a2e293dffae6dadef23a55f03c28b68d4ee4d6bd349b7a5abb1e2425",
 "q52": "0760bc163681565f05befb315abb3f6bd212c396769d07ed1546d6c212ca96cf",
 "q53": "44b9d064138aba255ded179991f283a231314e0903bed8ae3323daf63e3d6831",
 "q54": "78756e798c35826709fa467abd2f5e067fa6519b24ff02abdf400df2568deb77",
 "q55": "104ef068e4abb00f2f60b63cce9118c1b82a8a0674f0a399df7dda52fc69d774",
 "q56": "6473b9174624f03df55e2ca5eafc8aef21f84b3d388ae56ebe74fbbde8d6877a",
 "q57": "92249a60c4b8afcddb788644454144343dcf6849779c4e121611477bbe651e77",
 "q58": "1751c5edbff1e19674164cbec7e16fb0a2c58b1ff955dc0939b5796173ddd8b7",
 "q59": "5ca549888f3fa42c4732bfccbe12477f603326c8623a033f59eafcb896281823",
 "q60": "befe437c060787cb3678a9059f457e088befab9621bf9911c45128da362e70d6"
}
PASS_SCALED = 720


def _h(qid: str, letters: str) -> str:
    return hashlib.sha256(f"{qid}:{letters}".encode()).hexdigest()


def _show(md: str) -> None:
    if display is not None:
        display(Markdown(md))
    else:
        print(md)


def normalize(answer) -> str | None:
    """Accept "b", "B,D", "b d", ["B", "D"] or {"D", "B"}; return sorted unique letters, or None."""
    if answer is None:
        return None
    if isinstance(answer, (list, tuple, set)):
        answer = "".join(str(a) for a in answer)
    if not isinstance(answer, str):
        return None
    letters = sorted({c for c in answer.upper() if c.isalpha()})
    return "".join(letters) or None


def scaled_estimate(raw_fraction: float) -> int:
    """Approximation only: 0% -> 100, 72% -> 720, 100% -> 1000, linear in between."""
    p = min(max(raw_fraction, 0.0), 1.0)
    if p <= 0.72:
        return round(100 + p / 0.72 * 620)
    return round(720 + (p - 0.72) / 0.28 * 280)


def grade_exam(answers: dict, *, show_review: bool = True) -> dict:
    results, notes = {}, []
    for qid, meta in META.items():
        letters = normalize(answers.get(qid))
        if letters is None:
            results[qid] = False
            continue
        if len(letters) != meta["n"]:
            notes.append(f"{qid.upper()}: select exactly {meta['n']} (you gave {len(letters)})")
            results[qid] = False
            continue
        results[qid] = _h(qid, letters) == KEY_HASH[qid]
    correct = sum(results.values())
    total = len(META)
    raw = correct / total
    scaled = scaled_estimate(raw)
    unanswered = sum(1 for q in META if normalize(answers.get(q)) is None)

    print("CCAR-F mock exam · score report (study estimate, not official)")
    print("=" * 66)
    by_dom = {}
    for d in sorted(DOMAINS):
        qs = [q for q in META if META[q]["domain"] == d]
        ok = sum(results[q] for q in qs)
        by_dom[d] = (ok, len(qs))
        pct = ok / len(qs)
        print(f"Domain {d} ({WEIGHTS[d]:>2}%) {DOMAINS[d]:<40} {ok:>2}/{len(qs):<2} {pct:>4.0%}  {'█' * round(pct * 10)}{'░' * (10 - round(pct * 10))}")
    print("-" * 66)
    print(f"Raw score: {correct}/{total} = {raw:.1%}   (unanswered: {unanswered})")
    print(f"Scaled-score estimate: {scaled} / 1000   (pass = {PASS_SCALED})")
    print("Result:", "PASS" if scaled >= PASS_SCALED else "RETRY")
    start = globals().get("EXAM_START")
    if start:
        mins = (time.time() - start) / 60
        flag = "within" if mins <= 120 else "OVER"
        print(f"Time used: {mins:.0f} min ({flag} the 120-minute limit; about 2 min per item is the target)")
    print("Item grid: " + " ".join(f"{q[1:]}{'✓' if results[q] else '✗'}" for q in META))
    for n in notes:
        print("  note:", n)
    missed = [q for q in META if not results[q]]
    if show_review and missed:
        md = ["**Review list** (task statement → sections to reread):"]
        for q in missed:
            m = META[q]
            md.append(f"- Q{q[1:]} · Domain {m['domain']} · task {m['task']}: " + ", ".join(f"[{lb}]({u})" for lb, u in m["review"]))
        _show("\n".join(md))
    return {"correct": correct, "total": total, "raw": raw, "scaled": scaled,
            "passed": scaled >= PASS_SCALED, "by_domain": by_dom, "results": results}


report = grade_exam(answers)

## Scorecard: what to do next

| Raw score | Scaled estimate | Suggested next step |
|---|---|---|
| ≥ 80% (48+ of 60) **and** ≥ 70% in every domain | ≈ 800+ | Book the exam. Keep the drills warm and reread the explanations for every miss. |
| 72–79% (44–47), **or** 80%+ with any domain below 70% | ≈ 733–783 (or higher) | Borderline. Fix your weakest domains (their weeks in the study plan, then their drills), then book. |
| < 72% (43 or fewer) | < 720 | Not yet. Go back to the study-plan weeks for your weakest domains before booking. |

This is the same readiness rule as the [CCAR-F README](README.md#how-to-use-the-drills-and-the-mock) and the [exam prep index](../README.md#how-to-use-the-mocks) (a study heuristic, not an official threshold). The domain lines matter more than the total: a domain below 70% is a hole that a different set of four scenarios could expose. A second attempt at this same mock measures memory, not skill, so for a re-check use the [drills](01_practice.ipynb) you missed and the chapter homework quizzes.

Next: back to the [CCAR-F overview](README.md) to book your exam, or to the [Claude path overview](../../README.md).

---

## Explanations (spoilers)

Open these only after you have graded your attempt. Each explanation says why the right answer wins, why each distractor loses, and which section to reread.

<details><summary>Scenario A: Customer support resolution agent: answers and explanations (Q1–Q15)</summary>

**Key:** Q1 A · Q2 CD · Q3 D · Q4 B · Q5 A · Q6 A · Q7 B · Q8 B · Q9 A · Q10 D · Q11 B · Q12 A · Q13 B · Q14 B · Q15 C

**Q1: A** · Domain 1 · Task 1.5

**A** is right: a hook is your code and runs on every matching call, so the limit holds deterministically, and the deny reason redirects the agent to escalation.

Why not the others: **B**: still a prompt instruction, which lowers the violation rate but never guarantees zero; **C**: it fires after the credit is applied, so it detects violations instead of preventing them; **D**: examples improve consistency, but compliance stays probabilistic.

Review: [Ch 3 §3.5](../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk), [Ch 9 §9.4](../../09-escalation-human-in-the-loop/README.md#94-programmatic-enforcement-vs-prompt-guidance)

---

**Q2: C, D** · Domain 1 · Task 1.4

**C** is right: a programmatic gate in the agent process blocks the downstream calls until the prerequisite has really completed. **D** is right: enforcement on the server side is just as deterministic: the action cannot run without proof of verification.

Why not the others: **A**: self-reported confidence is not evidence of verification; **B**: prompt guidance has a non-zero failure rate, which is exactly the 4% you see; **E**: better, but still probabilistic for a rule with financial consequences.

Review: [Ch 9 §9.4](../../09-escalation-human-in-the-loop/README.md#94-programmatic-enforcement-vs-prompt-guidance), [Ch 3 §3.5](../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk)

---

**Q3: D** · Domain 1 · Task 1.1

**D** is right: `stop_reason` is the signal the API gives you; a response can contain text and a `tool_use` block together, so text is not a completion indicator.

Why not the others: **A**: an arbitrary iteration cap as the main stopping rule cuts off longer cases (a high cap is fine only as a safety guard); **B**: it treats the symptom with a probabilistic instruction and keeps the wrong termination check; **C**: parsing natural-language signals to end the loop is a named anti-pattern.

Review: [Ch 3 §3.1](../../03-agent-sdk/README.md#31-the-agentic-loop), [Ch 1 §1.3](../../01-claude-api-fundamentals/README.md#13-stop_reason-why-the-model-stopped), [Ch 2 §2.6](../../02-tool-use/README.md#26-the-client-side-tool-loop-and-tool_result-rules-expanded-beyond-the-guide)

---

**Q4: B** · Domain 1 · Task 1.4

**B** is right: the human cannot see the transcript, so everything needed to act must travel in the handoff itself.

Why not the others: **A**: the scenario says humans cannot access transcripts, and a transcript is not a summary anyway; **C**: it shifts the work to the customer and repeats the problem; **D**: escalation timing should follow escalation criteria, and an early escalation still arrives without context.

Review: [Ch 9 §9.6](../../09-escalation-human-in-the-loop/README.md#96-structured-handoff-protocols)

---

**Q5: A** · Domain 1 · Task 1.4

**A** is right: decompose, investigate with shared context, then synthesize: every concern is handled and the customer gets one coherent answer.

Why not the others: **B**: two of three issues stay unresolved; **C**: it pushes the decomposition onto the customer and hurts first-contact resolution; **D**: complexity alone is not an escalation trigger, and all three issues are within the agent's scope.

Review: [Ch 8 §8.9](../../08-task-decomposition/README.md#89-decomposing-multi-concern-requests), [Ch 9 §9.6](../../09-escalation-human-in-the-loop/README.md#96-structured-handoff-protocols)

---

**Q6: A** · Domain 1 · Task 1.5

**A** is right: normalizing in code once, before the model reads the data, removes the ambiguity deterministically.

Why not the others: **B**: still probabilistic, and `04/09/2026` is ambiguous no matter how carefully it reasons; **C**: the model still has to convert on every call, and it will sometimes get it wrong; **D**: adds latency and cost and is still model-based, so not guaranteed.

Review: [Ch 3 §3.5](../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk)

---

**Q7: B** · Domain 2 · Task 2.1

**B** is right: keyword-sensitive wording in the system prompt can create tool associations that beat the descriptions; fix the cause.

Why not the others: **A**: over-engineered, and it bypasses the model instead of fixing the prompt; **C**: a heavy redesign that leaves the misleading instruction in place; **D**: brittle keyword routing that breaks on phrasings like 'why did I pay more?'.

Review: [Ch 2 §2.2](../../02-tool-use/README.md#22-tool-definition)

---

**Q8: B** · Domain 2 · Task 2.2

**B** is right: the category and the retryable flag tell the agent whether to retry, fix the input, or explain a policy limit to the customer.

Why not the others: **A**: the agent gets no usable information and the customer loses the conversation; **C**: it hides the failure and invites the agent to report success; **D**: retrying a policy violation or a malformed ID can never succeed and wastes calls.

Review: [Ch 10 §10.4](../../10-multi-agent-error-handling/README.md#104-structured-error-responses-from-tools), [Ch 10 §10.2](../../10-multi-agent-error-handling/README.md#102-error-categories-and-the-retryable-flag), [Ch 4 §4.4](../../04-model-context-protocol/README.md#44-the-iserror-flag-and-structured-errors)

---

**Q9: A** · Domain 2 · Task 2.2

**A** is right: an access failure needs a retry decision; a valid empty result is a real answer. The two must look different.

Why not the others: **B**: wastes calls on genuine empties and still cannot separate the two cases; **C**: most empty results are legitimate answers the agent can handle; **D**: the agent still cannot tell which case it is in.

Review: [Ch 10 §10.3](../../10-multi-agent-error-handling/README.md#103-access-failure-vs-valid-empty-result)

---

**Q10: D** · Domain 2 · Task 2.3

**D** is right: more tools means a harder choice on every turn; a small role-scoped set restores selection reliability.

Why not the others: **A**: it forces a tool call, not the right tool; **B**: does not remove the cause: tools outside the agent's role; **C**: the decision space stays just as large, and the prompt grows.

Review: [Ch 2 §2.2](../../02-tool-use/README.md#22-tool-definition), [Ch 3 §3.2](../../03-agent-sdk/README.md#32-agentdefinition-configuration)

---

**Q11: B** · Domain 4 · Task 4.3

**B** is right: it combines two separate guarantees: forcing the tool makes Claude call it (no prose reply), and `strict: true` constrains its input to the schema. Without `strict`, a forced call still returns parsed JSON, but a field can have the wrong type or be missing. Before writing to the CRM, still check `stop_reason`: a `max_tokens` or `refusal` stop can leave the input incomplete. (Structured outputs via `output_config.format` give the same guarantee and are not among the options. Current docs: Claude Opus 5.5, Sonnet 5.5, Fable 5.1 and Mythos 5.1 reject `any` and forced `tool` with a 400; on those models use `auto` with `strict: true` plus a code check, or structured outputs.)

Why not the others: **A**: fragile post-processing of a problem you can prevent; **C**: truncation is only one possible cause; the format itself is still unenforced; **D**: a stronger instruction is still not a guarantee.

Review: [Ch 2 §2.3](../../02-tool-use/README.md#23-the-tool_choice-parameter), [Ch 2 §2.4](../../02-tool-use/README.md#24-json-schemas-for-structured-output)

---

**Q12: A** · Domain 4 · Task 4.2

**A** is right: targeted examples with reasoning teach the decision boundary and generalize to new phrasings.

Why not the others: **B**: a redesign for what is a judgment problem on a few phrasings; **C**: the agent already calls a tool; the problem is which one; **D**: clear cases already work; they cost tokens without teaching the ambiguous boundary.

Review: [Ch 6 §6.1](../../06-prompt-engineering/README.md#61-few-shot-prompting), [Ch 2 §2.2](../../02-tool-use/README.md#22-tool-definition)

---

**Q13: B** · Domain 5 · Task 5.2

**B** is right: an explicit request for a human is honored immediately.

Why not the others: **A**: it overrides an explicit request; **C**: sentiment is not the trigger; the explicit request is; **D**: still delays the escalation the customer asked for.

Review: [Ch 9 §9.2](../../09-escalation-human-in-the-loop/README.md#92-escalation-patterns), [Ch 9 §9.1](../../09-escalation-human-in-the-loop/README.md#91-when-to-escalate)

---

**Q14: B** · Domain 5 · Task 5.2

**B** is right: multiple matches call for clarification, not a heuristic guess.

Why not the others: **A**: premature: asking for one more identifier is within the agent's capability; **C**: discloses other customers' data; **D**: a heuristic guess risks acting on someone else's account.

Review: [Ch 9 §9.3](../../09-escalation-human-in-the-loop/README.md#93-resolving-ambiguity-ask-dont-guess)

---

**Q15: C** · Domain 5 · Task 5.1

**C** is right: transactional facts are extracted once and never pass through lossy summarization.

Why not the others: **A**: more summarization passes mean more chances to blur numbers and dates; **B**: postpones the loss instead of preventing it; **D**: shifts the burden to the customer and hurts the experience.

Review: [Ch 11 §11.1](../../11-context-management/README.md#111-extract-facts-into-a-persistent-case-facts-block)

</details>

<details><summary>Scenario B: Code generation with Claude Code: answers and explanations (Q16–Q30)</summary>

**Key:** Q16 C · Q17 D · Q18 B · Q19 C · Q20 B · Q21 AC · Q22 C · Q23 D · Q24 B · Q25 A · Q26 D · Q27 B · Q28 AE · Q29 C · Q30 DE

**Q16: C** · Domain 3 · Task 3.1

**C** is right: team rules belong at project level; user-level files apply only to their owner.

Why not the others: **A**: personal files are not shared through git and will drift; **B**: `/memory` shows and edits memory files; there is no team sync; **D**: `CLAUDE.local.md` is personal and meant to be git-ignored.

Review: [Ch 5 §5.1](../../05-claude-code/README.md#51-the-claudemd-hierarchy)

---

**Q17: D** · Domain 3 · Task 3.3

**D** is right: a path-scoped rule loads only when Claude reads or edits a file that matches its `paths`; the old pattern matches one folder, so the fix is the pattern, not the delivery mechanism.

Why not the others: **A**: a rule without `paths` loads at launch in every session, including the many that never touch a migration; **B**: the copies drift apart, and every new service needs another copy; **C**: depends on people remembering, so it is not automatic.

Review: [Ch 5 §5.3](../../05-claude-code/README.md#53-clauderules-topic-files-and-path-scoped-rules)

---

**Q18: B** · Domain 3 · Task 3.1

**B** is right: imports keep the files modular and let each maintainer choose what applies.

Why not the others: **A**: every package gets every standard, relevant or not; **C**: on-demand loading for standards that should always apply in that package; **D**: duplicates text, which then drifts.

Review: [Ch 5 §5.2](../../05-claude-code/README.md#52-path-imports-modular-claudemd)

---

**Q19: C** · Domain 3 · Task 3.2

**C** is right: the 2,000 lines of intermediate tool output stay in the subagent's context, and only the one-page summary comes back, so the main session keeps its focus. (Forking isolates the work, not the result: if the skill's final answer were itself 2,000 lines, it would still need to save details to a file and return a short summary.)

Why not the others: **A**: adds tokens to every session and does not isolate anything; **B**: lossy clean-up after the context is already polluted; **D**: controls who invokes it, not where its output goes.

Review: [Ch 5 §5.5](../../05-claude-code/README.md#55-skills-in-depth-skillmd-frontmatter), [Ch 11 §11.5](../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context)

---

**Q20: B** · Domain 3 · Task 3.2

**B** is right: personal skills live in the home directory, are not shared through git, and a distinct name avoids confusion with the team skill.

Why not the others: **A**: affects every teammate; **C**: project-scoped, so it ships to everyone who pulls; **D**: changes the shared skill for everyone.

Review: [Ch 5 §5.4](../../05-claude-code/README.md#54-custom-slash-commands--skills)

---

**Q21: A, C** · Domain 3 · Task 3.4

**A** is right: one small, well-understood change. **C** is right: a single-file fix with a clear cause is well scoped.

Why not the others: **B**: an architectural change with dependencies to discover; **D**: a multi-file migration benefits from exploring and planning first; **E**: several valid approaches with infrastructure trade-offs: plan first.

Review: [Ch 5 §5.6](../../05-claude-code/README.md#56-plan-mode-vs-direct-execution)

---

**Q22: C** · Domain 3 · Task 3.5

**C** is right: input/output examples communicate a transformation better than prose that is being read inconsistently.

Why not the others: **A**: plan mode is for exploring and designing changes, not for pinning down an output format; **B**: vague, and it depends on session history; **D**: more prose of the kind that is already being misread.

Review: [Ch 6 §6.9](../../06-prompt-engineering/README.md#69-matching-the-strategy-to-the-task-and-iterating-expanded-beyond-the-guide), [Ch 6 §6.1](../../06-prompt-engineering/README.md#61-few-shot-prompting)

---

**Q23: D** · Domain 1 · Task 1.7

**D** is right: resume when prior context is mostly valid, and name the changes for targeted re-analysis.

Why not the others: **A**: a fork copies the conversation as it is, stale tool results included, and refreshes nothing; you would still have to name the three files. Forking is for branching into divergent approaches from a shared baseline, which this task does not need; **B**: the history holds stale tool results that Claude will trust; **C**: throws away analysis that is still valid.

Review: [Ch 5 §5.10](../../05-claude-code/README.md#510-sessions-resume-name-fork), [Ch 11 §11.7](../../11-context-management/README.md#117-sessions-resume-fork-or-start-fresh-task-17)

---

**Q24: B** · Domain 1 · Task 1.7

**B** is right: each fork starts from the shared baseline and keeps its own history. Forks share the working tree, so use separate git worktrees if both branches will edit code.

Why not the others: **A**: the summary still carries the first exploration; **C**: repeats the expensive baseline work; **D**: the second exploration is biased by the first.

Review: [Ch 5 §5.10](../../05-claude-code/README.md#510-sessions-resume-name-fork), [Ch 3 §3.6](../../03-agent-sdk/README.md#36-sessions-resume-and-fork-bonus-from-domain-17)

---

**Q25: A** · Domain 1 · Task 1.6

**A** is right: an open-ended task needs dynamic, adaptive decomposition.

Why not the others: **B**: attention dilution, and no plan; **C**: file size is a poor proxy for impact; **D**: a fixed pipeline ignores risk and the dependencies you will discover.

Review: [Ch 8 §8.2](../../08-task-decomposition/README.md#82-dynamic-adaptive-decomposition)

---

**Q26: D** · Domain 5 · Task 5.4

**D** is right: persisted findings survive context degradation and compaction.

Why not the others: **A**: output length is unrelated to what the model has lost; **B**: degradation in long sessions is not only about window size; **C**: manual, error-prone, and you will drop details too.

Review: [Ch 11 §11.4](../../11-context-management/README.md#114-scratchpad-files)

---

**Q27: B** · Domain 5 · Task 5.4

**B** is right: verbose discovery stays out of the main context, which keeps the high-level picture.

Why not the others: **A**: still floods the main context with raw output; **C**: reactive and lossy compared with isolating the discovery; **D**: the fastest way to exhaust the context.

Review: [Ch 11 §11.5](../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context), [Ch 5 §5.6](../../05-claude-code/README.md#56-plan-mode-vs-direct-execution)

---

**Q28: A, E** · Domain 5 · Task 5.4

**A** is right: focus instructions tell the summary what to keep. **E** is right: a file is not touched by compaction and can be re-read.

Why not the others: **B**: summaries lose exact values first; **C**: `/clear` starts an empty context; **D**: a permission mode, unrelated to what the summary keeps.

Review: [Ch 5 §5.7](../../05-claude-code/README.md#57-compact-and-managing-the-context-window), [Ch 11 §11.4](../../11-context-management/README.md#114-scratchpad-files)

---

**Q29: C** · Domain 2 · Task 2.4

**C** is right: project scope shares the server through git; expansion keeps the secret out of the repo.

Why not the others: **A**: not shared, so configurations drift; **B**: secrets in git leak through forks, logs and history; **D**: a committed secret, and CLAUDE.md is context, not configuration.

Review: [Ch 4 §4.3](../../04-model-context-protocol/README.md#43-configuring-mcp-servers)

---

**Q30: D, E** · Domain 2 · Task 2.5

**D** is right: Grep searches file contents with ripgrep regex syntax, so the parenthesis is escaped (and `\s*` also catches a call written with a space before it). **E** is right: Glob matches file paths by pattern (it returns at most 100 paths, sorted by modification time, and flags a truncated result so Claude can narrow the pattern). (Current docs: on macOS, Linux and WSL, Claude Code leaves `Glob` and `Grep` out of the default tool set and searches with `find` and `grep` through `Bash`; naming them in `--tools`/`--allowedTools` or the Agent SDK options brings them back, and so does removing `Bash` from the session or giving a subagent a `tools` list that names them and leaves out `Bash`. The exam tests what each tool is for, as here.)

Why not the others: **A**: Edit changes one file by exact, unique text match; it takes no glob; **B**: Glob matches paths, not contents; **C**: it can work, but it is not the right tool: one Grep for the message text finds it directly, while reading everything wastes context.

Review: [Ch 13 §13.2](../../13-claude-code-builtin-tools/README.md#132-grep-vs-glob-content-search-vs-path-search), [Ch 13 §13.3](../../13-claude-code-builtin-tools/README.md#133-read-write-and-edit-full-file-vs-targeted-changes)

</details>

<details><summary>Scenario C: Multi-agent research system: answers and explanations (Q31–Q45)</summary>

**Key:** Q31 D · Q32 C · Q33 D · Q34 A · Q35 C · Q36 C · Q37 D · Q38 D · Q39 A · Q40 B · Q41 C · Q42 A · Q43 D · Q44 D · Q45 CE

**Q31: D** · Domain 1 · Task 1.2

**D** is right: dynamic selection by query complexity is part of the coordinator's job.

Why not the others: **A**: new simple questions still take the full path; **B**: synthesis depends on research, and simple queries still pay for everything; **C**: weakens complex reports without fixing simple ones.

Review: [Ch 3 §3.3](../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents), [Ch 8 §8.7](../../08-task-decomposition/README.md#87-workflow-or-agent-choosing-the-level-of-autonomy)

---

**Q32: C** · Domain 1 · Task 1.2

**C** is right: distinct assignments minimize duplicated work and widen coverage.

Why not the others: **A**: gives up parallel coverage; **B**: randomness is not a scope partition; **D**: the duplicated search work and cost remain.

Review: [Ch 3 §3.3](../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents)

---

**Q33: D** · Domain 1 · Task 1.2

**D** is right: an iterative refinement loop driven by the coordinator closes coverage gaps.

Why not the others: **A**: unsourced content in a cited report; **B**: repeats everything and will likely produce the same gap; **C**: ships a known gap the system could close.

Review: [Ch 3 §3.3](../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents), [Ch 8 §8.2](../../08-task-decomposition/README.md#82-dynamic-adaptive-decomposition)

---

**Q34: A** · Domain 1 · Task 1.3

**A** is right: a named subagent such as this one starts with a fresh context: the prompt is the only thing that travels from the coordinator to it. (Only a fork, a special subagent that copies the parent conversation, inherits context; this system does not use forks.)

Why not the others: **B**: there is no automatic shared memory to switch on; **C**: no model can use findings it never received; **D**: it would redo the analysis instead of using it, outside its role.

Review: [Ch 3 §3.4](../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task), [Ch 11 §11.5](../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context)

---

**Q35: C** · Domain 1 · Task 1.3

**C** is right: several subagent calls in one response run in parallel, so latency is set by the slowest.

Why not the others: **A**: removes the parallelism entirely; **B**: more turns do not make turns concurrent; **D**: still sequential, and it bypasses the coordinator.

Review: [Ch 3 §3.4](../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task)

---

**Q36: C** · Domain 1 · Task 1.3

**C** is right: goals and criteria let subagents adapt; rigid procedures break on variation.

Why not the others: **A**: retrying a step that does not fit the source fails again; **B**: the procedure gets more brittle, not less; **D**: even less adaptable.

Review: [Ch 3 §3.3](../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents), [Ch 3 §3.4](../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task)

---

**Q37: D** · Domain 1 · Task 1.2

**D** is right: hub-and-spoke keeps one place that sees, routes and recovers.

Why not the others: **A**: more uncontrolled paths, not fewer; **B**: loses specialization and still bypasses coordination; **C**: visibility after the fact, with no one responsible for recovery or routing.

Review: [Ch 3 §3.3](../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents), [Ch 10 §10.1](../../10-multi-agent-error-handling/README.md#101-where-failures-happen-in-a-multi-agent-system-expanded-beyond-the-guide)

---

**Q38: D** · Domain 2 · Task 2.3

**D** is right: a constrained tool makes the out-of-role use impossible instead of discouraged.

Why not the others: **A**: widens the misuse; **B**: detects instead of preventing; **C**: probabilistic, while the generic capability stays available.

Review: [Ch 2 §2.2](../../02-tool-use/README.md#22-tool-definition), [Ch 3 §3.2](../../03-agent-sdk/README.md#32-agentdefinition-configuration)

---

**Q39: A** · Domain 2 · Task 2.2

**A** is right: local recovery for transient failures; propagate only what cannot be resolved, with context.

Why not the others: **B**: one transient failure should not kill the workflow; **C**: silent suppression: the coordinator never learns about the gap; **D**: generic status hides context and still escalates what could be fixed locally.

Review: [Ch 10 §10.5](../../10-multi-agent-error-handling/README.md#105-local-recovery-before-propagation)

---

**Q40: B** · Domain 2 · Task 2.4

**B** is right: resources give agents a map of available content.

Why not the others: **A**: one giant tool result instead of a catalog; **C**: pays for the exploration instead of removing it; **D**: enormous context cost for a catalog problem.

Review: [Ch 4 §4.5](../../04-model-context-protocol/README.md#45-resources-give-the-agent-a-map)

---

**Q41: C** · Domain 4 · Task 4.2

**C** is right: examples of varied structures fix empty extractions where the information exists.

Why not the others: **A**: the method is exactly where no such heading exists; **B**: invites fabrication when a paper truly has no method; **D**: the same prompt on the same document gives the same result.

Review: [Ch 6 §6.1](../../06-prompt-engineering/README.md#61-few-shot-prompting)

---

**Q42: A** · Domain 4 · Task 4.3

**A** is right: it guarantees a tool call instead of prose while leaving the type decision to the model. Forcing a call is not schema validation: add `strict: true` to the three tools so their inputs also match the schemas. (Current docs: Claude Opus 5.5, Sonnet 5.5, Fable 5.1 and Mythos 5.1 reject `any` and forced `tool` with a 400; on those models use `auto` with `strict: true` plus a code check, or structured outputs.)

Why not the others: **B**: forces one schema onto all document types; **C**: `auto` allows a text reply, which is the current problem; **D**: forbids tool calls, so the JSON in the text has no schema behind it.

Review: [Ch 2 §2.3](../../02-tool-use/README.md#23-the-tool_choice-parameter)

---

**Q43: D** · Domain 5 · Task 5.6

**D** is right: conflicts are annotated with attribution, not silently resolved.

Why not the others: **A**: blocks the pipeline; finish the analysis with the conflict annotated; **B**: still silently picks a winner, and the method difference disappears; **C**: arbitrary selection that drops a credible source.

Review: [Ch 12 §12.2](../../12-provenance/README.md#122-handling-conflicting-data)

---

**Q44: D** · Domain 5 · Task 5.6

**D** is right: dates turn an apparent contradiction into a trend.

Why not the others: **A**: removes useful findings instead of interpreting them; **B**: invents a number that describes neither year; **C**: throws away the trend and the context.

Review: [Ch 12 §12.3](../../12-provenance/README.md#123-include-dates-for-correct-interpretation)

---

**Q45: C, E** · Domain 5 · Task 5.3

**C** is right: partial results plus an explicit gap is honest and still useful. **E** is right: coverage annotations tell readers how much weight each part can bear.

Why not the others: **A**: silent suppression of a known gap; **B**: one failure should not block the whole workflow; **D**: conflates an access failure with a valid empty result.

Review: [Ch 10 §10.9](../../10-multi-agent-error-handling/README.md#109-coverage-annotations-in-the-final-synthesis), [Ch 12 §12.6](../../12-provenance/README.md#126-coverage-and-uncertainty-reporting-expanded-beyond-the-guide), [Ch 10 §10.3](../../10-multi-agent-error-handling/README.md#103-access-failure-vs-valid-empty-result)

</details>

<details><summary>Scenario D: Claude Code for continuous integration: answers and explanations (Q46–Q60)</summary>

**Key:** Q46 A · Q47 AE · Q48 C · Q49 C · Q50 BC · Q51 B · Q52 D · Q53 C · Q54 C · Q55 A · Q56 BE · Q57 C · Q58 A · Q59 A · Q60 B

**Q46: A** · Domain 3 · Task 3.6

**A** is right: schema-validated output is machine-parseable by design.

Why not the others: **B**: a format instruction is not enforcement; **C**: still an instruction, now in a different file; **D**: a streaming envelope around the same unstructured text.

Review: [Ch 5 §5.9](../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode)

---

**Q47: A, E** · Domain 3 · Task 3.6

**A** is right: CLAUDE.md is how project context reaches CI runs. **E** is right: Claude can only avoid duplicates it can see. (Current docs recommend `--bare` for scripted runs and plan to make it the `-p` default; if you adopt it, pass the testing guidance explicitly, for example with `--append-system-prompt-file`.)

Why not the others: **B**: `--bare` skips CLAUDE.md, removing the guidance you need; **C**: changes cost and latency, not quality; **D**: more turns, same missing guidance.

Review: [Ch 5 §5.9](../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode)

---

**Q48: C** · Domain 3 · Task 3.6

**C** is right: prior findings in context let the review skip what is fixed and avoid duplicates.

Why not the others: **A**: loses the discussion history and still re-posts fixed issues; **B**: misses issues that remain unaddressed from earlier commits; **D**: wording varies between runs, so exact matching fails.

Review: [Ch 5 §5.9](../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode)

---

**Q49: C** · Domain 3 · Task 3.6

**C** is right: a reviewer without the generator's context is more likely to question its decisions.

Why not the others: **A**: two passes with the same bias; **B**: more thinking on top of the same biased context; **D**: self-review instructions do not remove the shared reasoning context.

Review: [Ch 8 §8.4](../../08-task-decomposition/README.md#84-independent-review-instances-why-self-review-is-weak), [Ch 5 §5.9](../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode)

---

**Q50: B, C** · Domain 3 · Task 3.6

**B** is right: print mode is the documented non-interactive mode. **C** is right: that metadata is what lets you resume, log and cost-track CI runs.

Why not the others: **A**: `-p` exists exactly for running without a terminal; **D**: `-p` runs load CLAUDE.md; only `--bare` skips it; **E**: the reverse: it is a print-mode flag.

Review: [Ch 5 §5.9](../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode)

---

**Q51: B** · Domain 4 · Task 4.1

**B** is right: an explicit, checkable criterion replaces a vague instruction.

Why not the others: **A**: confidence-based filtering does not define what counts as an issue; **C**: vague, so precision barely moves; **D**: an arbitrary cap drops real issues and keeps noise.

Review: [Ch 6 §6.2](../../06-prompt-engineering/README.md#62-explicit-criteria-vs-vague-instructions)

---

**Q52: D** · Domain 4 · Task 4.1

**D** is right: a high false-positive category undermines trust in the accurate ones; switch it off while you fix it.

Why not the others: **A**: adds friction and makes the noise more expensive; **B**: the criteria, not the model, produce the noise; **C**: uncalibrated confidence does not fix bad criteria.

Review: [Ch 6 §6.2](../../06-prompt-engineering/README.md#62-explicit-criteria-vs-vague-instructions)

---

**Q53: C** · Domain 4 · Task 4.2

**C** is right: contrast examples teach the boundary and generalize to similar patterns.

Why not the others: **A**: also loses the correct findings on unjustified disables; **B**: the model does not know these are false positives; **D**: a keyword filter that hides real issues too.

Review: [Ch 6 §6.1](../../06-prompt-engineering/README.md#61-few-shot-prompting)

---

**Q54: C** · Domain 4 · Task 4.4

**C** is right: a structured trigger field turns dismissals into data you can group and act on.

Why not the others: **A**: says that a finding was bad, not which construct caused it; **B**: hides findings without explaining the dismissals; **D**: hard to aggregate across thousands of findings.

Review: [Ch 6 §6.6](../../06-prompt-engineering/README.md#66-self-correction)

---

**Q55: A** · Domain 4 · Task 4.5

**A** is right: a merge that lands just after a submission waits up to one interval, then up to 24 hours of processing and 4 hours of handling: 8 + 24 + 4 = 36. (In production a request can also come back `expired` when its batch hits the 24-hour limit, so a real SLA design also needs a fallback, such as re-running expired requests through the synchronous API, and the time for it.)

Why not the others: **B**: a guarantee must use the 24-hour bound, not typical completion times: 16 + 24 + 4 = 44 hours; **C**: 12 + 24 = 36 already, which leaves no time for the 4 hours of validation and posting; **D**: worst case 24 + 24 + 4 = 52 hours.

Review: [Ch 7 §7.8](../../07-message-batches/README.md#78-sla-planning-and-submission-cadence)

---

**Q56: B, E** · Domain 4 · Task 4.5

**B** is right: `custom_id` correlates results with requests, so only the failures are re-run. **E** is right: the cause was size; resubmitting unchanged would fail again.

Why not the others: **A**: gives up the batch discount for a problem that chunking fixes; **C**: a context overflow is deterministic; **D**: pays again for 588 good results.

Review: [Ch 7 §7.6](../../07-message-batches/README.md#76-result-types-and-resubmitting-only-the-failures)

---

**Q57: C** · Domain 4 · Task 4.6

**C** is right: per-file passes find local issues; an integration pass finds cross-file ones.

Why not the others: **A**: longer output, same blind spot; **B**: brings back attention dilution and inconsistent depth; **D**: a per-file pass cannot see a cross-file bug, however often it runs.

Review: [Ch 8 §8.3](../../08-task-decomposition/README.md#83-multi-pass-code-review)

---

**Q58: A** · Domain 4 · Task 4.6

**A** is right: self-reported confidence becomes useful once it is calibrated on labeled data.

Why not the others: **B**: no measurement and no calibration; **C**: uncalibrated self-reports are unreliable; **D**: length is not a quality signal.

Review: [Ch 8 §8.5](../../08-task-decomposition/README.md#85-verification-passes-and-confidence-based-routing), [Ch 9 §9.7](../../09-escalation-human-in-the-loop/README.md#97-confidence-calibration)

---

**Q59: A** · Domain 2 · Task 2.3

**A** is right: forced selection guarantees the first call; later turns go back to normal choice. (Current docs: Claude Opus 5.5, Sonnet 5.5, Fable 5.1 and Mythos 5.1 reject `any` and forced `tool` with a 400; on those models use `auto` with `strict: true` plus a code check, or structured outputs.)

Why not the others: **B**: the order of tool definitions does not enforce anything; **C**: probabilistic; **D**: forces some tool, not this one.

Review: [Ch 2 §2.3](../../02-tool-use/README.md#23-the-tool_choice-parameter)

---

**Q60: B** · Domain 2 · Task 2.4

**B** is right: standard integrations are already solved; custom code is for what is unique to the team.

Why not the others: **A**: more code to build and maintain for no added capability; **C**: over-broad access with no tool contract; **D**: manual, and it turns project memory into per-PR scratch space.

Review: [Ch 4: Architect decision cheat-sheet](../../04-model-context-protocol/README.md#architect-decision-cheat-sheet)

</details>